# LangSmith 사용하기

## 환경 셋팅

In [1]:
# API key 로딩
from dotenv import load_dotenv
import os

load_dotenv(override=True)
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
LANGSMITH_API_KEY = os.environ["LANGSMITH_API_KEY"]

from langchain_openai import ChatOpenAI

# llm = ChatOpenAI(model="gpt-4o", api_key=OPENAI_API_KEY)
llm = ChatOpenAI(model="gpt-3.5-turbo-0125", api_key=OPENAI_API_KEY)

## 코드 실행

In [2]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import JsonOutputParser
from rich.console import Console
from rich.panel import Panel
import json

console = Console()

# 출력을 위한 JSON 스키마 정의
json_schema = """
{{
  "destination": "목적지",
  "duration": "기간",
  "overview": "여행 개요",
  "daily_plans": [
   {{
      "day": 1,
      "title": "제목",
      "morning": "오전 일정",
      "lunch": "점심 추천",
      "afternoon": "오후 일정",
      "dinner": "저녁 추천",
      "accommodation": "숙소 추천"
    }}
  ],
  "tips": ["팁1", "팁2"],
  "total_budget": "예상 총 비용"
}}
"""

# LLM 초기화
llm = ChatOpenAI(
    model="gpt-4o-mini", 
    temperature=0.5,
    model_kwargs={"response_format": {"type": "json_object"}}
)

# 프롬프트
prompt = ChatPromptTemplate.from_messages([
    ("system", f"""당신은 {{place}} 베테랑 여행 가이드입니다.
고객 최적의 {{place}} {{travel}} 일정 수립을 도와줍니다.

다음의 JSON 형식으로 응답하세요.
{json_schema}"""),
    ("human", "해당 장소 {place}의 {travel} 일정에 맞는 여행 계획을 제안해주세요.")
])

# Json 출력 파서 정의
output_parser = JsonOutputParser()

# 체인 구성
chain = prompt | llm | output_parser

# llm 호출 - 자동으로 LangSmith에 기록됨!
input_data = {
    "place": "서울",
    "travel": "3일"
}

console.print("[bold cyan]여행 계획 생성 중...[/bold cyan]")
result = chain.invoke(input_data)

# 결과 출력
console.print(Panel(
    json.dumps(result, indent=2, ensure_ascii=False),
    title="[bold green]여행 계획[/bold green]",
    border_style="green"
))

여행 계획 생성 중...

╭─────────────────────────────────────────────────── 여행 계획 ───────────────────────────────────────────────────╮
│ {                                                                                                               │
│   "destination": "서울",                                                                                        │
│   "duration": "3일",                                                                                            │
│   "overview": "서울의 전통과 현대가 어우러진 매력적인 여행 코스입니다. 역사적인 유적지 탐방과 현대적인 쇼핑,    │
│ 맛있는 음식들을 즐길 수 있습니다.",                                                                             │
│   "daily_plans": [                                                                                              │
│     {                                                                                                           │
│       "day": 1,                                                                                                 │
│       "title": "서울의 역사 탐방",                                                                              │
│       "morning": "경복궁 방문 및 광화문 광장 탐방",                                                             │
│       "lunch": "인사동의 전통 한정식 식당 추천",                                                                │
│       "afternoon": "북촌 한옥마을 산책 및 전통 공예 체험",                                                      │
│       "dinner": "종로의 유명한 불고기 집 추천",                                                                 │
│       "accommodation": "명동에 위치한 부티크 호텔 추천"                                                         │
│     },                                                                                                          │
│     {                                                                                                           │
│       "day": 2,                                                                                                 │
│       "title": "현대 서울과 쇼핑",                                                                              │
│       "morning": "남산타워 방문 및 서울 전경 감상",                                                             │
│       "lunch": "명동의 길거리 음식 탐방",                                                                       │
│       "afternoon": "동대문 디자인 플라자(DDP) 탐방 및 쇼핑",                                                    │
│       "dinner": "홍대의 트렌디한 퓨전 음식점 추천",                                                             │
│       "accommodation": "홍대 근처의 게스트하우스 추천"                                                          │
│     },                                                                                                          │
│     {                                                                                                           │
│       "day": 3,                                                                                                 │
│       "title": "문화와 자연 체험",                                                                              │
│       "morning": "창덕궁 방문 및 후원 탐방",                                                                    │
│       "lunch": "이화여대 근처의 카페에서 브런치",                                                               │
│       "afternoon": "서울숲 산책 및 자전거 대여",                                                                │
│       "dinner": "강남의 스시집 추천",                                                                           │
│       "accommodation": "강남의 비즈니스 호텔 추천"                                                              │
│     }                                                                                                           │
│   ],                                                                                                            │
│   "tips": [                                                                                                     │
│     "대중교통을 이용하면 서울의 모든 곳을 쉽게 이동할 수 있습니다.",                                            │


# google gemini 모니터링하기

In [ ]:
# 1. 필요한 패키지 설치
# pip install langchain-google-genai python-dotenv

import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from rich.console import Console
import json

load_dotenv(override=True)
console = Console()

# 2. 환경변수 설정 (.env 파일)
"""
GOOGLE_API_KEY=your-google-api-key
LANGCHAIN_TRACING_V2=true
LANGCHAIN_API_KEY=your-langsmith-api-key
LANGCHAIN_PROJECT=gemini-travel-planner
"""

# 3. JSON 스키마
json_schema = """
{{
  "destination": "목적지",
  "duration": "기간",
  "overview": "여행 개요",
  "daily_plans": [
   {{
      "day": 1,
      "title": "제목",
      "morning": "오전 일정",
      "lunch": "점심 추천",
      "afternoon": "오후 일정",
      "dinner": "저녁 추천",
      "accommodation": "숙소 추천"
    }}
  ],
  "tips": ["팁1", "팁2"],
  "total_budget": "예상 총 비용"
}}
"""

# 4. Gemini LLM 초기화 (OpenAI 대신)
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",  # 또는 "gemini-1.5-pro"
    temperature=0.5,
    # JSON 모드는 Gemini에서도 지원됨
)

# 5. 프롬프트
prompt = ChatPromptTemplate.from_messages([
    ("system", f"""당신은 {{place}} 베테랑 여행 가이드입니다.
고객 최적의 {{place}} {{travel}} 일정 수립을 도와줍니다.

다음의 JSON 형식으로 응답하세요.
{json_schema}"""),
    ("human", "해당 장소 {place}의 {travel} 일정에 맞는 여행 계획을 제안해주세요.")
])

# 6. 출력 파서
output_parser = JsonOutputParser()

# 7. 체인 구성
chain = prompt | llm | output_parser

# 8. 실행 (자동으로 LangSmith에 추적됨!)
console.print("[bold cyan]Gemini로 여행 계획 생성 중...[/bold cyan]")

result = chain.invoke(
    {"place": "서울", "travel": "3일"},
    config={
        "metadata": {
            "model": "gemini-2.5-flash",
            "user_id": "user-123"
        },
        "tags": ["gemini", "travel", "production"]
    }
)

# 9. 결과 출력
console.print("[bold green]✓ 완료![/bold green]")
console.print(json.dumps(result, indent=2, ensure_ascii=False))

console.print("\n[bold yellow]LangSmith에서 확인:[/bold yellow]")
console.print("https://smith.langchain.com")

Gemini로 여행 계획 생성 중...

✓ 완료!

{
  "destination": "서울",
  "duration": "3일",
  "overview": "서울의 과거와 현재, 다채로운 미식과 문화를 만끽할 수 있는 3일간의 맞춤형 일정입니다. 고궁의 
아름다움부터 최신 트렌드를 이끄는 번화가까지, 서울의 진정한 매력을 경험해 보세요.",
  "daily_plans": [
    {
      "day": 1,
      "title": "고궁과 한옥, 전통의 미를 찾아",
      "morning": "오전 9시, 조선의 정궁 '경복궁'을 방문하여 고궁의 웅장함을 느껴보세요. (한복 체험 강력 추천!) 
이어서 북촌 한옥마을을 거닐며 전통 한옥의 아름다움을 감상합니다.",
      "lunch": "삼청동 '수제비' 또는 서촌 '토속촌 삼계탕'에서 정갈한 한식으로 든든하게 점심 식사를 즐깁니다.",
      "afternoon": "인사동 쌈지길에서 한국 전통 공예품을 구경하고, 전통 찻집에서 여유로운 시간을 가져보세요. 
고즈넉한 '조계사' 방문으로 마음의 평화를 찾아봅니다.",
      "dinner": "종로 또는 익선동 한옥 거리에서 트렌디한 한식 다이닝(퓨전 한식 또는 한우)을 즐기며 전통과 현대가 
어우러진 분위기를 만끽합니다.",
      "accommodation": "명동 또는 종로 일대 (교통이 편리하고 주요 관광지에 대한 접근성이 우수합니다.)"
    },
    {
      "day": 2,
      "title": "서울의 심장, 쇼핑과 야경 만끽",
      "morning": "남산 케이블카를 타고 'N서울타워'에 올라 서울의 파노라마 전경을 감상합니다. 이후 '남산골 
한옥마을'을 둘러보며 도심 속 여유를 만끽합니다.",
      "lunch": "쇼핑의 메카 명동에서 다양한 길거리 음식 투어를 하거나, 백화점 푸드코트에서 취향에 맞는 메뉴를 
선택하여 즐깁니다.",
      "afternoon": "명동 거리에서 K-뷰티 제품 쇼핑과 한국 패션을 경험하고, 롯데백화점 본점 또는 신세계백화점 
본점에서 고급 쇼핑을 즐겨보세요.",
      "dinner": "명동에서 한국의 대표 메뉴인 '치맥(치킨과 맥주)'을 즐기거나, 남산 인근의 '돈까스' 맛집을 방문해 
보세요. 저녁에는 남산 타워나 명동의 루프탑 바에서 서울의 아름다운 야경을 감상하며 하루를 마무리합니다.",
      "accommodation": "명동 또는 종로 일대 (전날과 동일한 숙소 이용을 권장하여 이동의 번거로움을 줄입니다.)"
    },
    {
      "day": 3,
      "title": "문화 예술과 트렌디한 서울",
      "morning": "젊음과 예술의 거리 '홍대'를 방문하여 개성 넘치는 벽화와 독립 디자인 샵을 구경하고, 버스킹 공연을 
즐기며 활기찬 에너지를 느껴봅니다.",
      "lunch": "연남동 또는 홍대 인근의 개성 넘치는 브런치 카페나 세계 각국 요리 전문점에서 트렌디한 점심 식사를 
즐깁니다.",
      "afternoon": "합정 '메세나폴리스'에서 쇼핑 및 카페를 즐기거나, 상수동 '카페 거리'에서 여유로운 시간을 
보냅니다. (K-POP 팬이라면 YG 엔터테인먼트 사옥 근처 방문도 추천합니다.)",
      "dinner": "여행의 마지막 밤을 기념하여 강남 '가로수길' 또는 청담동에서 파인 다이닝을 경험하거나, 홍대에서 
자유롭게 클럽 문화를 체험하며 젊음의 밤을 즐겨보세요.",
      "accommodation": "여행 마무리 및 출국 준비 (숙소는 2일차와 동일하게 사용하며, 짐 정리 및 공항 이동 준비를 
합니다.)"
    }
  ],
  "tips": [
    "**교통:** '티머니 카드'를 구매하여 지하철과 버스를 이용하면 서울 주요 관광지에 편리하게 접근할 수 있습니다. 
'카카오T' 또는 '우티' 앱으로 택시 이용도 편리합니다.",
    "**내비게이션:** 구글 맵보다는 '네이버 지도'나 '카카오맵'이 한국 지리에 훨씬 정확하니 꼭 설치해서 활용하세요.",
    "**언어:** 기본적인 한국어 인사말 ('안녕하세요', '감사합니다')을 익혀두면 현지인들과 더 친밀한 교류를 할 수 
있습니다.",
    "**미식:** 명동, 광장시장 등에서 다양한 길거리 음식을 꼭 맛보세요! 서울의 진정한 맛을 경험할 수 있습니다.",
    "**예약:** 인기 있는 레스토랑이나 특정 투어(예: DMZ 투어)는 미리 예약하는 것이 좋습니다.",
    "**환전:** 주요 은행(우리, 신한, 국민 등)이나 명동 사설 환전소에서 환율 우대를 받을 수 있습니다. 필요 시 공항 
환전소도 이용 가능합니다.",
    "**개인 컵 지참:** 한국은 환경 보호를 위해 카페에서 텀블러 사용 시 할인을 해주는 곳이 많습니다. 개인 컵을 챙겨 
오시면 유용합니다."
  ],
  "total_budget": "1인당 약 60만원 ~ 120만원 (항공권 및 개인 쇼핑 제외, 숙소/식사/교통/입장료 기준. 개인의 소비 
성향과 숙소 등급에 따라 유동적으로 변동될 수 있습니다.)"
}

LangSmith에서 확인:

https://smith.langchain.com